In [1]:
import torch.nn as nn
import torch
import pandas as pd
import copy
import random
import numpy as np

In [2]:
max_len = 21
embed_dim = 256

In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
set_seed()

In [4]:
df = pd.read_csv('/Users/baonguyen/IU/thesis/data/clean_data/data_with_bertopic_column.csv')
df['review_date'] = pd.to_datetime(df['review_date'])
df['month'] = df['review_date'].dt.month


In [5]:
df = df.apply(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x), axis=0)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 192544 entries, 0 to 192543
Data columns (total 18 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   Unnamed: 0      192544 non-null  int64         
 1   fit             192544 non-null  object        
 2   user_id         192544 non-null  int64         
 3   bust size       192544 non-null  object        
 4   item_id         192544 non-null  int64         
 5   weight          192544 non-null  object        
 6   rating          192544 non-null  float64       
 7   rented for      192544 non-null  object        
 8   review_text     192544 non-null  object        
 9   body type       192544 non-null  object        
 10  review_summary  192544 non-null  object        
 11  category        192544 non-null  object        
 12  height          192544 non-null  object        
 13  size            192544 non-null  int64         
 14  age             192544 non-null  flo

In [6]:
df['review_date']=pd.to_datetime(df['review_date'])
df_sorted = df.sort_values('review_date')
df_sorted.rename(columns={'rented for':'rented_for','body type':'body_type','bust size':'bust_size'},inplace=True)

In [7]:
df_sorted.columns

Index(['Unnamed: 0', 'fit', 'user_id', 'bust_size', 'item_id', 'weight',
       'rating', 'rented_for', 'review_text', 'body_type', 'review_summary',
       'category', 'height', 'size', 'age', 'review_date', 'Topic', 'month'],
      dtype='object')

In [8]:
side_feature =  ['rented_for','Topic']

In [9]:
# item to index and vice versa
unique_item_id = set(df_sorted['item_id'])
item_to_index = {item:idx +1 for idx , item in enumerate(unique_item_id)}
index_to_item = {idx+1:item for idx , item in enumerate(unique_item_id)}

for i in side_feature:
    exec(f'unique_{i} = set(df_sorted[i])')
    exec(f'{i}_to_index = {{i:idx+1 for idx,i in enumerate(unique_{i})}}')
    exec(f'index_to_{i} = {{idx+1:i for idx,i in enumerate(unique_{i})}}')

In [10]:
# Step 1: Group and aggregate
user_item_sequence = (
    df_sorted.groupby('user_id')[['item_id']+side_feature]
    .agg(list)
    .to_dict(orient='index')
)

# Step 2: Remove users with fewer than 2 item_ids
user_item_sequence = {
    user: val
    for user, val in user_item_sequence.items()
    if len(val['item_id']) >= 2 and len(val['item_id'])<=21
}


In [11]:
user_item_to_index_sequence = {}

for user, value in user_item_sequence.items():
    user_dict = {
        'item_id': [item_to_index[item] for item in value['item_id']]
    }
    for i in side_feature:
        # Dynamically get the correct mapping dict by name
        mapping_dict = globals()[f"{i}_to_index"]
        user_dict[i] = [mapping_dict[a] for a in value[i]]
    user_item_to_index_sequence[user] = user_dict


In [12]:


def mask_sequence(sequence: dict, mask_ratio: float):
    labels = {}
    mask_seq = {}
    for user, seq in sequence.items():
        mask_seq[user] = copy.deepcopy(seq)  # Deep copy so original is untouched
        labels[user] = [-100] * len(seq['item_id'])
        for i in range(len(mask_seq[user]['item_id'])):
            if random.random() < mask_ratio:
                labels[user][i] = mask_seq[user]['item_id'][i]  # Save original item id
                mask_seq[user]['item_id'][i] = 0       # Mask the item id
                for feature in side_feature:
                    mask_seq[user][feature][i] = 0
                
    return mask_seq, labels


In [13]:

mask_seq , labels = mask_sequence(user_item_to_index_sequence,mask_ratio=0.35)


In [14]:
def padding(mask_seq, labels, max_len=64, pad_item = 0, pad_label=-100):
    """
    Pads all user sequences in mask_seq and labels to max_len.
    
    """
    def pad(seq, max_len, pad_value):
        if len(seq) < max_len:
            return seq + [pad_value] * (max_len - len(seq))
        else:
            return seq[len(seq)-max_len:len(seq)]
    
    padded_mask_seq = {}
    padded_labels = {}

    for user in mask_seq:
        padded_mask_seq[user] = {
            **{'item_id': pad(mask_seq[user]['item_id'], max_len, pad_item)},
            **{i: pad(mask_seq[user][i], max_len, pad_item) for i in side_feature}
        }

        padded_labels[user] = pad(labels[user], max_len, pad_label)
    
    return padded_mask_seq, padded_labels


In [15]:
# padded_mask_seq,padded_labels = padding(mask_seq,labels,max_len=max_len)

In [16]:
# import numpy as np
# class SinusoidalPositionalEncoding(nn.Module):
#     def __init__(self, hidden_size, max_len=5000):
#         super(SinusoidalPositionalEncoding, self).__init__()
#         position = torch.arange(0, max_len).unsqueeze(1)
#         div_term = torch.exp(torch.arange(0, hidden_size, 2) * -(np.log(10000.0) / hidden_size))
#         pe = torch.zeros(max_len, hidden_size)
#         pe[:, 0::2] = torch.sin(position * div_term)
#         pe[:, 1::2] = torch.cos(position * div_term)
#         pe = pe.unsqueeze(0)
#         self.register_buffer('pe', pe)
#     def forward(self, x):
#         seq_len = x.size(1)
#         return self.pe[:, :seq_len, :]


In [17]:
# tensor_item_ids = torch.stack([
#     torch.tensor(user_seq['item_id']) for user_seq in padded_mask_seq.values()
# ])
# tensor_topic_ids = torch.stack([
#     torch.tensor(user_seq['Topic']) for user_seq in padded_mask_seq.values()
# ])
# tensor_labels = torch.stack([
#     torch.tensor(seq) for seq in padded_labels.values()
#     ])
# train_dataset = torch.utils.data.TensorDataset(
#     tensor_item_ids,
#     tensor_topic_ids,
#     tensor_labels
# )
# train_dataloader = torch.utils.data.DataLoader(train_dataset,batch_size=32,shuffle=True)

In [18]:
class differentiable_attn_mask(nn.Module):
    def __init__(self,):
        super(differentiable_attn_mask).__init__()
    pass

# nova bert architecture

In [19]:
class GatingFusor(nn.Module):
    def __init__(self, h):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(h, 1))

    def forward(self, features):   
        gates = torch.sigmoid(features @ self.weight)         
        fused = torch.sum(gates * features, dim=2)            
        return fused

In [20]:
class NovabertEmbedding(nn.Module):
    def __init__(self,num_item,num_side_feature_ids:dict,embedding_dim,max_len=64):
        super(NovabertEmbedding,self).__init__()
        self.item_embedding = nn.Embedding(num_item,embedding_dim)
        self.side_embedding_  = nn.ModuleDict()
        for feat_name,num_feat in num_side_feature_ids.items():
            self.side_embedding_[feat_name]=nn.Embedding(num_feat,embedding_dim)
        self.position_encoding = nn.Embedding(max_len, embedding_dim)
    def forward(self, item_ids, side_feature_ids: dict):
        position_ids = torch.arange(item_ids.size(1), dtype=torch.long, device=item_ids.device)
        position_ids = position_ids.unsqueeze(0).expand_as(item_ids)
        item_embed = self.item_embedding(item_ids)
        pos_embed = self.position_encoding(position_ids)
        item_embed = item_embed + pos_embed
        side_emb_list = []

        for feat_name in self.side_embedding_:
            
            feat_ids = side_feature_ids[feat_name]   # <-- Access by key, get tensor
            side_embed = self.side_embedding_[feat_name](feat_ids) + pos_embed
            side_emb_list.append(side_embed)
            
        return item_embed, side_emb_list


In [21]:
class NovabertCrossAttention(nn.Module):
    def __init__(self,embedding_dim,num_heads=8):
        super(NovabertCrossAttention,self).__init__()
        self.num_heads = num_heads
        self.head_dim = embedding_dim //num_heads
        self.value_proj = nn.Linear(embedding_dim,embedding_dim)
        self.query_proj = nn.Linear(embedding_dim,embedding_dim)
        self.key_proj = nn.Linear(embedding_dim,embedding_dim)
        self.fusor = GatingFusor(h=embedding_dim)


        self.output_proj = nn.Sequential(
              
        )
    def forward(self,item_embed,side_feature_embed:list,attn_mask=None,key_padding_mask=None):
        batch_size , sequence_len , embedding_dim = item_embed.size()
        def reshape(x:torch.tensor):
            return x.view(batch_size,sequence_len,self.num_heads,self.head_dim).transpose(1,2)
        # Batch,num_head,sequence_len,head_dim  (B,H,L,D)
        # print(*side_feature_embed)
        
        
        features = torch.stack([item_embed] + side_feature_embed, dim=2)

        fused_features = self.fusor(features)
        V = self.value_proj(item_embed)
        Q = self.query_proj(fused_features)
        K = self.key_proj(fused_features)
        Q = reshape(Q)
        K = reshape(K)
        V = reshape(V)
        scores = torch.matmul(Q,K.transpose(-2,-1)) / np.sqrt(self.head_dim) # B,H,L,L 
        
        if attn_mask is not None:
            scores += attn_mask.unsqueeze(0)  # Broadcast across batch ??? **********
            pass
        if key_padding_mask is not None:
            key_padding_mask = key_padding_mask.unsqueeze(1).unsqueeze(2) # B,1,1,L
            scores = scores.masked_fill(key_padding_mask,float('-inf'))
            # print(scores)

        attn_weights = torch.softmax(scores,dim=-1)
        attn_weights = torch.nan_to_num(attn_weights, nan=0.0)
        attn_output = torch.matmul(attn_weights,V) 

        # concat
        attn_output = attn_output.transpose(1,2).contiguous().view(batch_size,sequence_len,embedding_dim)
        return self.output_proj(attn_output)

            

In [22]:
class NovabertLayer(nn.Module):
    def __init__(self,embedding_dim,num_heads):
        super(NovabertLayer,self).__init__()
        self.cross_attn = NovabertCrossAttention(embedding_dim, num_heads)
        self.ffn = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim * 4),
            nn.GELU(),
            nn.Linear(embedding_dim * 4, embedding_dim)
        )
        self.norm1 = nn.LayerNorm(embedding_dim)
        self.norm2 = nn.LayerNorm(embedding_dim)
        self.dropout = nn.Dropout(0.2)
    def forward(self, id_embed,side_feature_embed:list , attention_mask=None,key_padding_mask = None):
        guided = self.cross_attn(id_embed,side_feature_embed,attention_mask,key_padding_mask)
        x = self.norm1(id_embed + self.dropout(guided))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x

In [23]:
class NovabertModel(nn.Module):
    def __init__(self, num_items,num_side_feature_ids:dict, embedding_dim, max_len=64, num_layers=4, num_heads=8):
        super(NovabertModel, self).__init__()
        self.embedding = NovabertEmbedding(num_item=num_items,
                                           num_side_feature_ids=num_side_feature_ids,
                                           embedding_dim=embedding_dim,
                                           max_len=max_len)
        self.nova_layer = nn.ModuleList([
            NovabertLayer(embedding_dim, num_heads) 
            for _ in range(num_layers)
        ])
        
        self.output_layer = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(embedding_dim, num_items)
        )

    def forward(self, item_ids, side_feature_ids: dict, attention_mask=None, key_padding_mask=None):
        x, y = self.embedding(item_ids, side_feature_ids)
        for layer in self.nova_layer:
            x = layer(x, y, attention_mask, key_padding_mask)
        return self.output_layer(x)



In [24]:
def precision_at_k(ground_truth: list, prediction: list, k: int):
    precisions = []
    for gt_item, pred in zip(ground_truth, prediction):
        recommended = pred[:k]
        hit = 1 if gt_item in recommended else 0
        precisions.append(hit / k)
    return sum(precisions) / len(precisions)

def recall_at_k(ground_truth: list, prediction: list, k: int):
    recalls = []
    for gt_item, pred in zip(ground_truth, prediction):
        recommended = pred[:k]
        hit = 1 if gt_item in recommended else 0
        recalls.append(hit)
    return sum(recalls) / len(recalls)

def mrr(ground_truth: list, prediction: list):
    rr = []
    for gt_item, pred in zip(ground_truth, prediction):
        if gt_item in pred:
            rank = pred.index(gt_item) + 1
            rr.append(1.0 / rank)
        else:
            rr.append(0.0)
    return sum(rr) / len(rr)

import math

def ndcg_at_k(ground_truth: list, prediction: list, k: int):
    ndcgs = []
    for gt_item, pred in zip(ground_truth, prediction):
        if gt_item in pred[:k]:
            rank = pred.index(gt_item) + 1
            dcg = 1 / math.log2(rank + 1)
            idcg = 1.0  # since only one ground truth item
            ndcgs.append(dcg / idcg)
        else:
            ndcgs.append(0.0)
    return sum(ndcgs) / len(ndcgs)

def coverage(prediction: list, catalog: set):
    recommended_items = set(item for user_pred in prediction for item in user_pred)
    return len(recommended_items) / len(catalog)

In [25]:
def hit_ratio(ground_truth:list,prediction:list,k:int):
    hits = 0
    total = len(ground_truth)
    for i, (gt_item, pred) in enumerate(zip(ground_truth, prediction)):
        
        if gt_item in pred[:k]:
            print(f"[Sample {i}] GT: {gt_item}, Pred top-{k}: {pred[:k]}")
            hits += 1
    return hits / total
# -------------------------
# Function to load the popularity data (counts.csv)
def load_popularity_data(filepath):
    df = pd.read_csv(filepath)
    item_popularity = dict(zip(df['item_id'], df['count']))  # Item popularity dictionary
    total_count = sum(item_popularity.values())  # Total count of interactions
    item_probabilities = {item: count / total_count for item, count in item_popularity.items()}  # Normalize probabilities
    return item_popularity, item_probabilities
# -------------------------
# Function to sample negative items based on popularity
def sample_negatives_by_popularity(all_items, item_probabilities, num_negatives=100, interacted_item=None):
    """Sample N negative items based on popularity, excluding the ground truth."""
    possible_negatives = all_items - set(interacted_item)
    negatives = np.random.choice(
    a=list(possible_negatives),                                    # candidates
    size=min(num_negatives, len(possible_negatives)),              # sample size
    replace=False,                                                 # no duplicates
    p=np.array([item_probabilities.get(item, 0) 
                for item in possible_negatives], dtype=float) / 
      max(1e-12, sum(item_probabilities.get(item, 0) 
                     for item in possible_negatives))              # normalize weights
).tolist()
    
    return negatives
# -------------------------
item_popularity, item_probabilities = load_popularity_data('/Users/baonguyen/IU/thesis/data/counts.csv')
item_probabilities = {item_to_index[key]:value for key,value in item_probabilities.items()}
all_items = [i for i in range(len(unique_item_id)+1)]
all_items=set(all_items)
def evaluate_model(model, val_item_sequences, k=10, max_len=64,  index_to_item=None, device='mps'):
    """
    Evaluate model hit ratio@k on validation data.

    Args:
        model: The trained model.
        val_item_sequences: Dict of user_id -> {'item_id': [...], <feat1>: [...], <feat2>: [...], ...}
        k: Top-k for hit ratio.
        max_len: Sequence length for padding.
        side_feature: List of feature names, e.g. ['Topic', 'category'].
        index_to_item: Dict mapping item indices back to original item ids.
        device: Device to run the model on.

    Returns:
        Hit ratio@k.
        precision_at_k.
        recall_at_k.
        mrr.
        ndcg_at_k.
        coverage.
    """
    model.eval()
    ground_truths = []
    predictions = []

    with torch.no_grad():
        for user, seq in val_item_sequences.items():
            item_seq = seq['item_id']
            # Prepare input and target
            input_items = item_seq[:-1]
            target_item = item_seq[-1]
            # Pad input sequence (handle all side features)
            mask_seq = {user: {'item_id': input_items}}
            for feat in (side_feature or []):
                mask_seq[user][feat] = seq[feat][:-1]
            padded_seq, _ = padding(
                mask_seq=mask_seq,
                labels={user: []},
                max_len=max_len
            )
            padded_items = padded_seq[user]['item_id']
            # Prepare side feature tensors as a dict
            side_input_dict = {
                feat: torch.tensor([padded_seq[user][feat]], dtype=torch.long).to(device)
                for feat in (side_feature or [])
            }
            item_tensor = torch.tensor([padded_items], dtype=torch.long).to(device)
            key_padding_mask = (item_tensor == 0)
            # Model call
            logits = model(item_tensor, side_input_dict, key_padding_mask=key_padding_mask)[:, min(len(item_seq)-1, max_len-1), :]
            probabilities = torch.softmax(logits, dim=-1)
            # --- Popularity-based Negative Sampling ---
            # Sample N negative items (those not interacted with by the user)
            negatives = sample_negatives_by_popularity(all_items, item_probabilities, num_negatives=100, interacted_item=item_seq)
            candidates = [target_item] + negatives

            # Get probabilities for the candidate items only
            candidate_logits = probabilities[0, candidates]  # Shape: (N+1,)
            
            # Rank candidates by their logits (probabilities)
            ranked = [x for _, x in sorted(zip(candidate_logits.tolist(), candidates), reverse=True)]

            # Store the ground truth and top-k predictions
            ground_truths.append(index_to_item[target_item])
            predictions.append([index_to_item[i] for i in ranked])

            # print(ground_truths)
            # print(predictions)
    return hit_ratio(ground_truths, predictions, k), \
            precision_at_k(ground_truths, predictions, k), \
            recall_at_k(ground_truths, predictions, k), \
              mrr(ground_truths, predictions), \
              ndcg_at_k(ground_truths, predictions, k), \
              coverage(predictions, set(index_to_item.values()))


In [26]:
import torch
import torch.optim as optim
from tqdm import tqdm
import os

epoch_num = 10
hitrate = 5

def train_model(
    train_users, val_users, fold_num,
    user_item_to_index_sequence, side_feature, unique_item_id,
    embedding_dim=256, max_len=max_len, num_layers=2, num_heads=4, mask_ratio=0.5
):
    # 1) Build training tensors
    train_user_item_to_index_sequence = {user: seq for user, seq in user_item_to_index_sequence.items() if user in train_users}
    mask_seq, labels = mask_sequence(train_user_item_to_index_sequence, mask_ratio=mask_ratio)
    padded_mask_seq, padded_labels = padding(mask_seq, labels, max_len=max_len)

    tensor_item_ids = torch.stack([
        torch.tensor(user_seq['item_id']) for user_seq in padded_mask_seq.values()
    ])
    tensor_side_feats = {
        feat: torch.stack([
            torch.tensor(user_seq[feat]) for user_seq in padded_mask_seq.values()
        ])
        for feat in side_feature
    }
    tensor_labels = torch.stack([torch.tensor(seq) for seq in padded_labels.values()])

    train_dataset = torch.utils.data.TensorDataset(
        tensor_item_ids,
        *(tensor_side_feats[feat] for feat in side_feature),
        tensor_labels
    )
    train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

    # 2) Model / optim
    device = torch.device('mps' if torch.backends.mps.is_available()
                          else 'cuda' if torch.cuda.is_available() else 'cpu')
    num_side_feature_ids = {feat: len(globals()[f'unique_{feat}']) for feat in side_feature}
    model = NovabertModel(
        len(unique_item_id) + 1,
        num_side_feature_ids=num_side_feature_ids,
        embedding_dim=embedding_dim,
        max_len=max_len,
        num_layers=num_layers,
        num_heads=num_heads
    ).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=0.001)
    criterion = torch.nn.CrossEntropyLoss(ignore_index=-100)

    # 3) Tracking best + history
    best = {
        "epoch": -1,
        "HR": -1.0,
        "Precision": 0.0,
        "Recall": 0.0,
        "MRR": 0.0,
        "NDCG": 0.0,
        "Coverage": 0.0,
    }
    history = []  # per-epoch metrics

    # 4) Train loop
    for epoch in range(epoch_num):
        model.train()
        epoch_loss = 0.0

        for batch in tqdm(train_dataloader, desc=f"Fold {fold_num} Epoch {epoch+1}", unit="batch"):
            item_ids = batch[0]
            side_ids = {feat: batch[i + 1] for i, feat in enumerate(side_feature)}
            labels = batch[-1]

            item_ids = item_ids.to(device)
            side_ids = {feat: tensor.to(device) for feat, tensor in side_ids.items()}
            labels = labels.to(device)

            key_padding_mask = (item_ids == 0)
            optimizer.zero_grad()
            outputs = model(item_ids, side_ids, key_padding_mask=key_padding_mask)  # [B, T, V]
            loss = criterion(outputs.view(-1, len(unique_item_id) + 1), labels.view(-1))
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss {epoch_loss:.4f}")

        # 5) Evaluate
        model.eval()
        val_item_sequences = {u: seq for u, seq in user_item_to_index_sequence.items() if u in val_users}
        val_hr, val_precision_at_k, val_recall_at_k, val_mrr, val_ndcg_at_k, val_coverage = evaluate_model(
            model,
            val_item_sequences,
            k=hitrate,
            max_len=max_len,
            index_to_item=index_to_item,
            device=device
        )

        print(f"Fold {fold_num} Epoch {epoch+1}, Validation HR@{hitrate}: {val_hr:.4f}")
        print(f"Fold {fold_num} Epoch {epoch+1}, Validation Precision@{hitrate}: {val_precision_at_k:.4f}")
        print(f"Fold {fold_num} Epoch {epoch+1}, Validation Recall@{hitrate}: {val_recall_at_k:.4f}")
        print(f"Fold {fold_num} Epoch {epoch+1}, Validation MRR: {val_mrr:.4f}")
        print(f"Fold {fold_num} Epoch {epoch+1}, Validation NDCG@{hitrate}: {val_ndcg_at_k:.4f}")
        print(f"Fold {fold_num} Epoch {epoch+1}, Validation Coverage: {val_coverage:.4f}")

        # log history
        history.append({
            "epoch": epoch + 1,
            "loss": float(epoch_loss),
            "HR": float(val_hr),
            "Precision": float(val_precision_at_k),
            "Recall": float(val_recall_at_k),
            "MRR": float(val_mrr),
            "NDCG": float(val_ndcg_at_k),
            "Coverage": float(val_coverage),
        })

        # 6) Save checkpoint if HR improves (you can change the selection rule)
        if val_hr > best["HR"]:
            best.update({
                "epoch": epoch + 1,
                "HR": float(val_hr),
                "Precision": float(val_precision_at_k),
                "Recall": float(val_recall_at_k),
                "MRR": float(val_mrr),
                "NDCG": float(val_ndcg_at_k),
                "Coverage": float(val_coverage),
            })

            save_path = f"models/models_item_with_novabert_gatingfusor_/fold_{fold_num}"
            os.makedirs(save_path, exist_ok=True)
            torch.save(model.state_dict(), f"{save_path}/best_model.pth")

    # 7) Return the model, best metrics, and full history
    return model, best, history


In [27]:
from sklearn.model_selection import KFold
import os
import torch
import json

kf = KFold(n_splits=5, shuffle=True, random_state=42)
user_list = list(user_item_sequence.keys())

# Store full metrics per fold (not just HR)
fold_metrics = {}   # {fold_num: {"epoch":..,"HR":..,"Precision":..,"Recall":..,"MRR":..,"NDCG":..,"Coverage":..}}
histories = {}      # optional: keep per-epoch history if you want to inspect/plot later

for fold_num, (train_idx, val_idx) in enumerate(kf.split(user_list), 1):
    print(f"\nStarting Fold {fold_num}...")
    train_users = [user_list[i] for i in train_idx]
    val_users   = [user_list[i] for i in val_idx]

    # Train model on this fold (returns best metrics & epoch history)
    model, best_metrics, history = train_model(
        train_users=train_users,
        val_users=val_users,
        fold_num=fold_num,
        user_item_to_index_sequence=user_item_to_index_sequence,
        side_feature=side_feature,
        unique_item_id=unique_item_id,
    )

    fold_metrics[fold_num] = best_metrics
    histories[fold_num] = history  # keep if you want; remove this line if not needed

    # Save per-fold results
    save_path = f"results/results_item_with_novabert_gatingfusor_/fold_{fold_num}"
    os.makedirs(save_path, exist_ok=True)
    with open(f"{save_path}/results.txt", "w") as f:
        f.write(f"Best Epoch: {best_metrics['epoch']}\n")
        f.write(f"HR@{hitrate}:        {best_metrics['HR']:.4f}\n")
        f.write(f"Precision@{hitrate}: {best_metrics['Precision']:.4f}\n")
        f.write(f"Recall@{hitrate}:    {best_metrics['Recall']:.4f}\n")
        f.write(f"MRR:                  {best_metrics['MRR']:.4f}\n")
        f.write(f"NDCG@{hitrate}:      {best_metrics['NDCG']:.4f}\n")
        f.write(f"Coverage:             {best_metrics['Coverage']:.4f}\n")

    # (Optional) also save per-epoch history for later analysis
    with open(f"{save_path}/history.json", "w") as f:
        json.dump(history, f, indent=2)

    # Free memory
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif hasattr(torch, 'mps') and torch.backends.mps.is_available():
        torch.mps.empty_cache()

# ---- Write overall summary (means across folds) ----
overall_path = "results/results_item_with_novabert_gatingfusor_/overall_results.txt"
os.makedirs(os.path.dirname(overall_path), exist_ok=True)

metric_names = ["HR", "Precision", "Recall", "MRR", "NDCG", "Coverage"]

with open(overall_path, "w") as f:
    for fold in sorted(fold_metrics.keys()):
        m = fold_metrics[fold]
        f.write(
            f"Fold {fold} (Best Epoch {m['epoch']}):\n"
            f"  HR@{hitrate}:        {m['HR']:.4f}\n"
            f"  Precision@{hitrate}: {m['Precision']:.4f}\n"
            f"  Recall@{hitrate}:    {m['Recall']:.4f}\n"
            f"  MRR:                  {m['MRR']:.4f}\n"
            f"  NDCG@{hitrate}:      {m['NDCG']:.4f}\n"
            f"  Coverage:             {m['Coverage']:.4f}\n\n"
        )

    f.write("===========================\n")
    f.write("Mean Metrics Across Folds:\n")
    f.write("===========================\n\n")
    for name in metric_names:
        vals = [fold_metrics[fold][name] for fold in sorted(fold_metrics.keys())]
        mean_val = float(sum(vals) / len(vals)) if vals else 0.0
        f.write(f"Mean {name}: {mean_val:.4f}\n")

# Console printout of means
print("\n✅ Training finished across all folds:")
for name in metric_names:
    vals = [fold_metrics[f][name] for f in sorted(fold_metrics.keys())]
    mean_val = float(sum(vals) / len(vals)) if vals else 0.0
    print(f"Mean {name}: {mean_val:.4f}")



Starting Fold 1...


Fold 1 Epoch 1: 100%|██████████| 419/419 [00:23<00:00, 17.88batch/s]


Epoch 1, Loss 3429.2082
[Sample 3] GT: 450618, Pred top-5: [123793, 1729232, 450618, 125465, 1687082]
[Sample 4] GT: 450618, Pred top-5: [2260466, 708493, 450618, 1679420, 1191124]
[Sample 81] GT: 174086, Pred top-5: [174086, 137585, 172027, 126335, 132738]
[Sample 126] GT: 123793, Pred top-5: [174086, 137585, 126335, 123793, 136860]
[Sample 127] GT: 313568, Pred top-5: [1514308, 313568, 501426, 422368, 2331652]
[Sample 136] GT: 131117, Pred top-5: [174086, 126335, 132738, 131117, 125465]
[Sample 159] GT: 132738, Pred top-5: [174086, 137585, 126335, 172027, 132738]
[Sample 172] GT: 136860, Pred top-5: [174086, 126335, 168592, 136860, 136110]
[Sample 191] GT: 174086, Pred top-5: [174086, 137585, 172027, 123793, 132738]
[Sample 203] GT: 136110, Pred top-5: [126335, 136860, 132738, 136110, 131117]
[Sample 264] GT: 2546911, Pred top-5: [2546911, 1729232, 1715008, 1459539, 534314]
[Sample 299] GT: 123793, Pred top-5: [174086, 126335, 123793, 132738, 136110]
[Sample 334] GT: 145906, Pred top

Fold 1 Epoch 2: 100%|██████████| 419/419 [00:25<00:00, 16.70batch/s]


Epoch 2, Loss 3278.7014
[Sample 3] GT: 450618, Pred top-5: [174086, 125465, 450618, 1226293, 125424]
[Sample 4] GT: 450618, Pred top-5: [1106101, 1968677, 450618, 1571668, 932152]
[Sample 24] GT: 1076484, Pred top-5: [126335, 174086, 127865, 1076484, 137585]
[Sample 75] GT: 136110, Pred top-5: [126335, 174086, 172027, 136110, 137585]
[Sample 81] GT: 174086, Pred top-5: [172027, 174086, 136110, 137585, 125465]
[Sample 126] GT: 123793, Pred top-5: [126335, 127865, 136110, 123793, 132738]
[Sample 152] GT: 136110, Pred top-5: [127865, 126335, 136110, 174086, 1076484]
[Sample 159] GT: 132738, Pred top-5: [126335, 174086, 127865, 131533, 132738]
[Sample 191] GT: 174086, Pred top-5: [172027, 174086, 921642, 131533, 137585]
[Sample 203] GT: 136110, Pred top-5: [174086, 136110, 125465, 168592, 145906]
[Sample 209] GT: 450618, Pred top-5: [921642, 450618, 127865, 1213427, 123793]
[Sample 225] GT: 1991314, Pred top-5: [468020, 890105, 858304, 295362, 1991314]
[Sample 259] GT: 2859490, Pred top-5:

...

Epoch 8, Loss 2940.2612
[Sample 36] GT: 580060, Pred top-5: [263699, 174086, 580060, 1210233, 123793]
[Sample 63] GT: 136110, Pred top-5: [136110, 174086, 123793, 166633, 131533]
[Sample 65] GT: 132738, Pred top-5: [126335, 123793, 131117, 136110, 132738]
[Sample 77] GT: 1457171, Pred top-5: [703458, 1459683, 1992625, 1457171, 1523882]
[Sample 81] GT: 868096, Pred top-5: [868096, 1706067, 1788074, 1976130, 1766932]
[Sample 83] GT: 241461, Pred top-5: [1076484, 241461, 166633, 1226293, 987569]
[Sample 85] GT: 1328587, Pred top-5: [2511676, 488181, 1295171, 288472, 1328587]
[Sample 88] GT: 786827, Pred top-5: [1224461, 786827, 599262, 2590191, 2239596]
[Sample 104] GT: 123793, Pred top-5: [174086, 144051, 123793, 136110, 166006]
[Sample 107] GT: 126335, Pred top-5: [126335, 137585, 128959, 130259, 147594]
[Sample 112] GT: 124204, Pred top-5: [1186923, 131117, 127865, 124204, 1076484]
[Sample 139] GT: 168592, Pred top-5: [126335, 131117, 168592, 123793, 136110]
[Sample 145] GT: 1460606, P

Fold 5 Epoch 9: 100%|██████████| 419/419 [00:20<00:00, 20.62batch/s]


Epoch 9, Loss 2862.8380
[Sample 3] GT: 166633, Pred top-5: [136860, 172027, 123793, 166633, 131117]
[Sample 27] GT: 746366, Pred top-5: [127865, 1939936, 522755, 130259, 746366]
[Sample 63] GT: 136110, Pred top-5: [174086, 137585, 136110, 131533, 126335]
[Sample 77] GT: 1457171, Pred top-5: [301873, 1774524, 902478, 1457171, 1238932]
[Sample 81] GT: 868096, Pred top-5: [2856326, 1793377, 868096, 1871370, 962489]
[Sample 83] GT: 241461, Pred top-5: [127865, 241461, 152836, 883661, 131533]
[Sample 86] GT: 2231364, Pred top-5: [549751, 2410432, 232082, 546688, 2231364]
[Sample 88] GT: 786827, Pred top-5: [344877, 786827, 1692512, 1033454, 376219]
[Sample 93] GT: 1529884, Pred top-5: [877767, 1529884, 834814, 597314, 497719]
[Sample 104] GT: 123793, Pred top-5: [174086, 137585, 144051, 127865, 123793]
[Sample 107] GT: 126335, Pred top-5: [174086, 126335, 131533, 132738, 131117]
[Sample 134] GT: 1819243, Pred top-5: [1795313, 1871026, 1507056, 1188641, 1819243]
[Sample 142] GT: 1191124, Pre

Fold 5 Epoch 10: 100%|██████████| 419/419 [00:19<00:00, 21.01batch/s]


Epoch 10, Loss 2761.1068
[Sample 63] GT: 136110, Pred top-5: [136110, 174086, 127495, 128730, 172027]
[Sample 77] GT: 1457171, Pred top-5: [1457171, 1693615, 1148823, 127865, 1723747]
[Sample 81] GT: 868096, Pred top-5: [1083818, 2273596, 868096, 276603, 124204]
[Sample 88] GT: 786827, Pred top-5: [1804544, 1679360, 786827, 989029, 2560442]
[Sample 93] GT: 1529884, Pred top-5: [1274956, 1523882, 1529884, 409531, 124204]
[Sample 101] GT: 152836, Pred top-5: [126335, 127865, 152836, 144051, 125465]
[Sample 104] GT: 123793, Pred top-5: [127865, 126335, 136110, 172027, 123793]
[Sample 107] GT: 126335, Pred top-5: [126335, 174086, 127865, 144051, 137585]
[Sample 112] GT: 124204, Pred top-5: [298336, 124204, 2793818, 1598962, 1408079]
[Sample 134] GT: 1819243, Pred top-5: [1967775, 1819243, 2667547, 1596783, 1271853]
[Sample 139] GT: 168592, Pred top-5: [126335, 174086, 152836, 132738, 168592]
[Sample 142] GT: 1191124, Pred top-5: [1797448, 222318, 1191124, 2269659, 2531493]
[Sample 157] GT: